# 当前 v4 的完整 30 样本开发回归

日期：2026-10-30。批次：`formal_30_v4_full_regression`。本记录在本批首次模型调用前建立。

复用原 v3 的 30 个组件、原始顺序和哈希一致的 Vue 输入，运行最终 v4 系统；简单／中等／复杂各 10 项，最多 3 轮 repair。本批是已接触样本的开发回归。原 v3 结果保留为 18 通过、9 失败、3 执行错误；本批不得覆盖该历史结果。

TaskManager 最近一次单组件自动及人工通过使用了 6 轮 repair；其结论不能替代本批的 3 轮整批结果。原 v4 的 12 项开发终态也不等于最终代码已通过全部 30 项。

协议：`data/experiments/protocols/formal_30_v4_full_regression_frozen_20261030/PROTOCOL.md`。配置：`migration_pipeline/batch_configs/formal_30_v4_full_regression.json`。预运行脚本：`python -m migration_pipeline.prepare_formal_30_v4_full_regression --freeze`（只执行本地校验与封存）。

v5 探索性评估等待本次开发回归完成；若再修改系统实现，v5 需要重新建立冻结版本。

## 运行与结果口径

重启本地服务并核对协议 live 校验后，由操作者运行：

```powershell
python migration_pipeline/run_batch_pipeline.py `
  --config-file migration_pipeline/batch_configs/formal_30_v4_full_regression.json `
  --include-state
```

不加 `--force`，不筛选 sample-id。基础设施中断按相同命令恢复。保留失败与执行错误，先冻结原始结果再诊断。自动结果和人工交互检查分开记录。

逐样本比较保持通过、回归失败、改善通过与仍未通过，另列执行异常。由于评估器也曾改变，结果差异不能全部归因于 Prompt。

In [ ]:
# 只读查看当前进度；本单元不运行迁移，也不改写实验报告。
import json
from pathlib import Path

root = Path.cwd().resolve()
while not (root / 'migration_pipeline').is_dir() and root != root.parent:
    root = root.parent
summary_path = root / 'data/experiments/batch_reports/formal_30_v4_full_regression_summary.json'
if summary_path.is_file():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(json.dumps(summary['aggregate'], ensure_ascii=False, indent=2))
else:
    print('本批尚无运行 summary，当前仅为预运行记录。')

## 首次整批执行结果与冻结证据（2026-10-30）

运行时间 20:20:54—21:20:53 +08:00。30 项均已尝试：25 通过、3 项评估失败、2 项执行错误，28 项正常完成。以全部配置样本为分母是 **25/30 = 83.33%**；正常完成项口径为 25/28 = 89.29%。不能写成整批通过。

原始结果先冻结于 `data/experiments/snapshots/formal_30_v4_full_regression_results_20261030_215511`，359 个文件校验通过；磁盘代码与预运行协议的 251 个 live 校验项通过。原始 summary、所有已保存产物和逐样本报告均未因分析而改写。

| 复杂度 | 原 v3 通过／配置数 | 当前 v4 通过／配置数 | 当前评估失败 | 当前执行错误 |
| --- | ---: | ---: | ---: | ---: |
| simple | 10/10 | 8/10 | 2 | 0 |
| medium | 7/10 | 10/10 | 0 | 0 |
| complex | 1/10 | 7/10 | 1 | 2 |

同样本配对：16 保持通过、9 改善为通过、2 由通过变未通过（BudgetThreshold、TipSplitCalculator）、3 仍未通过或执行错误（LocalizationWorkspace、TaskManager、MonitoringDashboard）。整体比原 v3 多通过 7 项，描述性变化 +23.33 个百分点；评估器与模型采样也会影响变化，不能全部归因于 Prompt。

summary 用量为生成 650,037、repair 1,092,872、共 1,742,909 tokens，repair 43 轮；只统计正常完成项，未完整包括异常样本已发生的用量。不能当作全批实际成本。全部 28 个完成项没有自动交互评估结论。

## 未通过项诊断与下一步

- **BudgetThreshold**：初始 DOM 相似度 1.0；首次 repair 将模板改成 `this.data.get('remaining')` 与 `this.data.get('statusText')`，渲染警告报告 this.data 未定义，最终结构相似度 0.75。3 轮耗尽；本批评分保留失败。
- **TipSplitCalculator**：DOM 相似度 1.0；静态 `custom_event_not_fire` 两轮无进展。源组件 `$emit` 只位于未被内部调用的通用 emitEvent 辅助方法，生成代码移除了它。需审查公开方法契约与校验器覆盖范围，当前不能直接断言界面事件丢失；原评分保留失败。
- **LocalizationWorkspace**：3 轮后仍有 `hidden_template_reactive_dependency`，模板 translationFor(record) 内部读取 language 却未显式传入；初始 DOM 一致不能证明语言切换响应正确。
- **TaskManager**：repair 请求发生上游读超时；初始及两份 repair 产物保留，但无完整最终报告。它是执行错误，并非已经验证为 3 轮修复失败。
- **MonitoringDashboard**：generation 请求发生上游读超时，执行约 240.75 秒，无最终评估。

下一步按原命令、不加 --force 进行基础设施恢复：checkpoint 会跳过全部 28 条 completed 记录（含三个评估失败），只重试两条 error。TaskManager 会重新执行样本链路，并非接着上次 repair 步继续。保持模型、Prompt、轮数、超时与实现不变，重试后的结果再建立新快照；本次快照与分析保留。

完整 30 项清单与错误证据：`data/experiments/analysis/formal_30_v4_full_regression_20261030_215511/REPORT.md`、`summary.json`。分析仅使用冻结证据：`python -m migration_pipeline.analyze_formal_30_v4_full_regression --snapshot data/experiments/snapshots/formal_30_v4_full_regression_results_20261030_215511`。v5 探索性评估继续等待；本轮没有修改 Prompt、迁移规则、评估器或生成 San 文件。

## 同协议超时重试仍未完成（2026-10-30）

用户按相同命令、不加 --force 恢复 checkpoint。TaskManager 重试时间为 22:50:13—22:54:14，耗时 241.047 秒；MonitoringDashboard 为 22:54:14—22:58:15，耗时 240.703 秒。两项均在 generation 请求发生 `The read operation timed out`。TaskManager 首次是 repair 超时，本次在生成阶段更早中断，不能因此判定修复规则退化。

原始重试状态冻结于 `data/experiments/snapshots/formal_30_v4_full_regression_results_20261030_231238`，359 个文件校验通过；首次快照 359 个文件亦复核通过。只改变了两个 error 样本记录，其余 28 条 completed 记录逐字段不变。pipeline／repair report 文件哈希对比没有差异，目录中保留的 TaskManager 产物仍来自首次运行，不能视为本次重试产物。

整批维持 **25 通过、3 评估失败、2 执行错误，25/30 = 83.33%**。28 个完成项的 token 聚合仍为 1,742,909，但不完整包含异常请求成本；不能把聚合未变解释为重试没有消耗。

冻结代码与协议设置表明：上游非流式请求的 LLM_TIMEOUT 是 240 秒，批处理到本地服务器的 generation／repair 客户端超时为 300 秒。两次约 241 秒的错误符合上游读取超时边界；现有日志无法确定具体是上游排队、生成慢还是网络读等待。

后续应保留本批原分数，单独准备更长超时的诊断版本（建议上游 600 秒、两类客户端 660 秒，模型／Prompt／3 轮上限保持），采用新协议、新输出与新 checkpoint；这项参数变化尚未实施。三个评估失败继续作为后续系统修复事项，v5 探索性评估等待。

重试对照报告：`data/experiments/analysis/formal_30_v4_full_regression_retry_20261030_231238/REPORT.md`、`summary.json`。复现分析：`python -m migration_pipeline.analyze_formal_30_v4_full_regression_retry --snapshot data/experiments/snapshots/formal_30_v4_full_regression_results_20261030_231238`。本次仅封存、诊断和追加记录，没有修改生成文件、Prompt、规则或超时，也未调用模型。